<a href="https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yumna-09/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

if not os.path.isdir("FlyRank-ML-Internship"):
    !git clone https://github.com/yumna-09/FlyRank-ML-Internship.git

os.chdir("/content/FlyRank-ML-Internship")
print(os.getcwd())

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = ["impressions_90d", "ctr", "avg_position", "engagement_rate", "content_age_days"]
X = df[features].dropna()

X.head()

Cloning into 'FlyRank-ML-Internship'...
remote: Enumerating objects: 160, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 160 (delta 63), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (160/160), 1.88 MiB | 15.76 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/FlyRank-ML-Internship


,impressions_90d,ctr,avg_position,engagement_rate,content_age_days
0,3803,0.76,10.6,5.88,187
1,15320,0.05,20.3,0.00,445
2,12581,0.09,36.5,0.00,141
3,11751,0.49,6.2,1.28,463
4,19140,0.13,44.0,0.00,263


I went with KMeans clustering for this task because my lane doesn't have a label to predict — I just need to group pages that behave similarly based on impressions, CTR, position, engagement, and content age. KMeans is a good starting point because it's simple, fast, and easy to explain. I picked the number of clusters using the silhouette score instead of guessing a number randomly.

In [18]:
expected_ctr_by_tier = df[(df["impressions_90d"] >= 500) & (df["position_tier"] != "no_data") & (df["avg_position"] > 0)].groupby("position_tier")["ctr"].median()

eligible = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["position_tier"] != "no_data")
df["expected_ctr"] = df["position_tier"].map(expected_ctr_by_tier)
df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)

def percentile_rank(s):
    return s.rank(method="average", pct=True)

df["ctr_gap_norm"] = percentile_rank(df["ctr_gap"].where(eligible, 0.0))
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["baseline_action_score"] = np.where(eligible, df["visibility_score"] * df["ctr_gap_norm"], 0.0).round(4)

def action_label(score):
    if score >= 0.6: return "prioritize_ctr_review"
    if score > 0: return "review_ctr"
    return "monitor"

df["action"] = df["baseline_action_score"].apply(action_label)
print(df["action"].value_counts())

action
monitor                  13274
review_ctr               12814
prioritize_ctr_review     3912
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Mean:", X_scaled.mean(axis=0))
print("Std:", X_scaled.std(axis=0))

Mean: [ 1.90662301e-17 -2.06057393e-17 -1.46371804e-16 -1.08949886e-17
 -2.84217094e-18]
Std: [1. 1. 1. 1. 1.]


Clustering doesn't use labels, so a normal train/test split doesn't apply the same way it does for classification. Instead, I scaled all my features first (using StandardScaler) so that large-scale numbers like impressions_90d don't overpower smaller-scale ones like ctr. After scaling, the mean of every feature was ~0 and the standard deviation was 1, confirming the scaling worked correctly.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scores = {}
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

print(scores)

best_k = max(scores, key=scores.get)
print("Best k:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df.loc[X.index, "cluster"] = kmeans.fit_predict(X_scaled)
final_score = silhouette_score(X_scaled, kmeans.labels_)
print("KMeans silhouette score:", final_score)

baseline_labels = df.loc[X.index, "action"]
baseline_score = silhouette_score(X_scaled, baseline_labels)
print("Baseline (rule-based action buckets) silhouette score:", baseline_score)

comparison = pd.DataFrame({
    "Method": ["Baseline (Week 4 rule-based buckets: monitor/review_ctr/prioritize_ctr_review)",
               f"KMeans clustering (k={best_k})"],
    "Silhouette Score": [baseline_score, final_score]
})
comparison

{2: np.float64(0.31895878651047194), 3: np.float64(0.3316033369344526), 4: np.float64(0.3495718557668614), 5: np.float64(0.3588668917460479), 6: np.float64(0.3681556013123113), 7: np.float64(0.37629582132673645)}
Best k: 7
KMeans silhouette score: 0.37629582132673645
Baseline (rule-based action buckets) silhouette score: 0.0013389962793651502


,Method,Silhouette Score
0,Baseline (Week 4 rule-based buckets: monitor/r...,0.001339
1,KMeans clustering (k=7),0.376296


I tried k values from 2 to 7 and calculated the silhouette score for each. The score kept increasing across that whole range (0.319 at k=2 up to 0.376 at k=7), so k=7 gave the best result. For a fair comparison, I treated my Week 4 baseline — the rule-based action buckets (monitor / review_ctr / prioritize_ctr_review) — as a "baseline grouping" and calculated its silhouette score on the same scaled features. The baseline scored just 0.0013, essentially no meaningful separation, while KMeans with k=7 scored 0.376. This makes sense: the Week 4 buckets were built around a single rule (CTR gap vs. position), not around how pages naturally group across all five features together, so they were never going to separate cleanly in this feature space. KMeans clearly finds more natural, well-separated groups than the rule-based buckets did.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import silhouette_samples

cluster_profile = df.groupby("cluster")[features].mean().round(2)
cluster_profile["count"] = df.groupby("cluster").size()
print(cluster_profile)

sample_scores = silhouette_samples(X_scaled, kmeans.labels_)
df.loc[X.index, "silhouette_val"] = sample_scores

weak_points = df.loc[X.index].sort_values("silhouette_val").head(10)
print("\nWorst-fit points (model most 'wrong' about these):")
print(weak_points[features + ["cluster", "silhouette_val"]])

print(f"\n% of points with negative silhouette (badly misplaced): {(sample_scores < 0).mean()*100:.1f}%")

         impressions_90d    ctr  avg_position  engagement_rate  \
cluster                                                          
0.0              3935.13   0.42         11.23             1.53   
1.0              3991.06   0.29         11.59             1.23   
2.0               403.31   3.73         18.15            97.34   
3.0                 4.20  41.35          6.23             6.14   
4.0              2786.01   0.10         46.20             0.92   
5.0              1720.59   0.56         15.34            28.86   
6.0            112096.49   0.34         10.68             3.09   

         content_age_days  count  
cluster                           
0.0                382.59  10023  
1.0                150.22  14247  
2.0                291.08    102  
3.0                286.54    132  
4.0                309.19   4134  
5.0                270.36    947  
6.0                261.11    415  

Worst-fit points (model most 'wrong' about these):
       impressions_90d   ctr  avg_posi

Looking at the average profile of each cluster, they roughly represent distinct content archetypes: Cluster 6 is a small group of very high-traffic pages (avg 112k impressions_90d) with typical CTR and good position — the site's top performers. Cluster 1 and 0 are the largest groups — steady, older content with moderate impressions and low engagement, differing mainly by age (Cluster 1 is much newer, ~150 days vs Cluster 0's ~383 days). Cluster 4 is low-CTR, poorly-ranked content (avg position 46) that isn't getting much visibility. Cluster 2 and 3 are small, unusual groups — Cluster 3 has almost no impressions but very high CTR (likely low-volume, highly targeted queries), and Cluster 2 has unusually high engagement relative to its traffic.

Checking per-point silhouette scores, about 2.6% of pages are poorly matched to their assigned cluster (negative silhouette). Almost all of these weak-fit points landed in Cluster 5 — pages with an engagement_rate near 16.67, which sits right between Cluster 1's low engagement (~1.2) and Cluster 2's very high engagement (~97). These borderline pages don't clearly belong to either the "low engagement" or "high engagement" pattern, which is exactly where a hard clustering method like KMeans struggles — it has to force every point into exactly one group even when a page's behavior is genuinely in between two archetypes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.